# APH/PPH Split

This notebook checks the APH/PPH split and hemorrhage logic:

1. APH and PPH incidence are in the ballpark of artifact expectations
2. Only severe hemorrhage cases die from hemorrhage
3. Misoprostol affects postpartum hemorrhage incidence but not antepartum hemorrhage incidence

In [15]:
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import math

import vivarium_gates_mncnh
from vivarium import Artifact, InteractiveContext
from vivarium.framework.configuration import build_model_specification

from vivarium_gates_mncnh.constants.data_keys import MATERNAL_HEMORRHAGE
from vivarium_gates_mncnh.constants.data_values import (
    ANEMIA_THRESHOLDS,
    COLUMNS,
    HEMORRHAGE_SEVERITY,
    SIMULATION_EVENT_NAMES,
    SIMULATION_STEPS,
)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

In [16]:
# Build a small interactive simulation for quick checks
model_spec_path = Path(vivarium_gates_mncnh.__file__).parent / 'model_specifications/model_spec.yaml'
base_spec = build_model_specification(model_spec_path)

DRAW = int(base_spec.configuration.input_data.input_draw_number)
POP_SIZE = 120_000

STEP_MAPPER = {name: i + 1 for i, name in enumerate(SIMULATION_STEPS)}

def make_spec(scenario: str = 'baseline', pop_size: int = POP_SIZE):
    spec = deepcopy(base_spec)
    spec.configuration.population.population_size = pop_size
    spec.configuration.intervention.scenario = scenario
    return spec

def make_sim(scenario: str = 'baseline', pop_size: int = POP_SIZE):
    return InteractiveContext(make_spec(scenario=scenario, pop_size=pop_size))

# TODO tdy fix this to use a while loop
def run_to_step(sim: InteractiveContext, step_name: str):
    sim.take_steps(STEP_MAPPER[step_name])
    return sim

artifact = Artifact(base_spec.configuration.input_data.artifact_path)
draw_col = f'draw_{DRAW}'
print('Using artifact:', base_spec.configuration.input_data.artifact_path)
print('Using draw column:', draw_col)
print('Population size:', POP_SIZE)

Using artifact: /mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/aph/ethiopia.hdf
Using draw column: draw_60
Population size: 120000


## 1) APH/PPH incidence and severity vs artifact

In [3]:
def get_art_by_age(art_key: str, sim_year: int, sim_location: str, sim_sex: str = 'Female') -> pd.DataFrame:
    art = artifact.load(art_key).reset_index()

    # Filter progressively; if a filter wipes everything out, keep previous data
    cur = art.copy()
    if 'location' in cur.columns:
        next_df = cur[cur['location'] == sim_location]
        if not next_df.empty:
            cur = next_df
    if 'sex' in cur.columns:
        next_df = cur[cur['sex'] == sim_sex]
        if not next_df.empty:
            cur = next_df
    if {'year_start', 'year_end'}.issubset(cur.columns):
        next_df = cur[(cur['year_start'] <= sim_year) & (sim_year < cur['year_end'])]
        if not next_df.empty:
            cur = next_df

    grouped = cur.groupby(['age_start', 'age_end'], as_index=False)[draw_col].mean()
    return grouped.sort_values(['age_start', 'age_end']).reset_index(drop=True)

def map_ages_to_art_values(ages: pd.Series, art_by_age: pd.DataFrame) -> pd.Series:
    intervals = pd.IntervalIndex.from_arrays(
        art_by_age['age_start'].astype(float),
        art_by_age['age_end'].astype(float),
        closed='left',
    )
    idx = intervals.get_indexer(ages.astype(float))
    out = pd.Series(np.nan, index=ages.index, dtype=float)
    matched = idx >= 0
    if matched.any():
        out.loc[matched] = art_by_age.iloc[idx[matched]][draw_col].to_numpy()
    return out

def weighted_incidence_from_artifact(
    pop: pd.DataFrame,
    art_key: str,
    eligible_mask: pd.Series,
    sim_year: int,
    sim_location: str,
    sim_sex: str = 'Female',
) -> float:
    art_by_age = get_art_by_age(art_key, sim_year=sim_year, sim_location=sim_location, sim_sex=sim_sex)
    eligible_ages = pop.loc[eligible_mask, COLUMNS.MOTHER_AGE]
    values = map_ages_to_art_values(eligible_ages, art_by_age)
    matched = values.notna()
    return float(values[matched].mean()) if matched.any() else np.nan

sim_year = int(base_spec.configuration.time.start.year)

# Single run for both APH and PPH checks (up to PPH step)
sim = make_sim('baseline')
run_to_step(sim, SIMULATION_EVENT_NAMES.POSTPARTUM_HEMORRHAGE)
pop = sim.get_population([
    COLUMNS.MOTHER_AGE,
    COLUMNS.LOCATION,
    COLUMNS.PREGNANCY_OUTCOME,
    COLUMNS.ANTEPARTUM_HEMORRHAGE,
    COLUMNS.POSTPARTUM_HEMORRHAGE,
])

sim_location = pop[COLUMNS.LOCATION].mode().iloc[0]

# Denominators must match component eligibility rules
aph_eligible = pop[COLUMNS.PREGNANCY_OUTCOME] != 'invalid'
pph_eligible = pop[COLUMNS.PREGNANCY_OUTCOME].isin(['live_birth', 'stillbirth'])

# Sim incidence
aph_cases = pop[COLUMNS.ANTEPARTUM_HEMORRHAGE].astype(bool)
pph_cases = pop[COLUMNS.POSTPARTUM_HEMORRHAGE].astype(bool)

aph_incidence_sim = float(aph_cases[aph_eligible].mean()) if int(aph_eligible.sum()) else np.nan
pph_incidence_sim = float(pph_cases[pph_eligible].mean()) if int(pph_eligible.sum()) else np.nan

# Artifact incidence weighted to simulated eligible age composition
aph_incidence_art = weighted_incidence_from_artifact(
    pop,
    MATERNAL_HEMORRHAGE.APH_INCIDENCE_RISK,
    aph_eligible,
    sim_year=sim_year,
    sim_location=sim_location,
)
pph_incidence_art = weighted_incidence_from_artifact(
    pop,
    MATERNAL_HEMORRHAGE.PPH_INCIDENCE_RISK,
    pph_eligible,
    sim_year=sim_year,
    sim_location=sim_location,
)

check1 = pd.DataFrame([
    {
        'cause': 'antepartum_hemorrhage',
        'sim_incidence': aph_incidence_sim,
        'artifact_incidence': aph_incidence_art,
        'incidence_ratio_sim_over_artifact': aph_incidence_sim / aph_incidence_art if aph_incidence_art else np.nan,
    },
    {
        'cause': 'postpartum_hemorrhage',
        'sim_incidence': pph_incidence_sim,
        'artifact_incidence': pph_incidence_art,
        'incidence_ratio_sim_over_artifact': pph_incidence_sim / pph_incidence_art if pph_incidence_art else np.nan,
    },
])

check1

2026-06-16 10:20:52.134 | INFO     | simulation_1-artifact_manager:80 - Running simulation from artifact located at /mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/aph/ethiopia.hdf.
2026-06-16 10:20:52.136 | INFO     | simulation_1-artifact_manager:81 - Artifact base filter terms are ['draw == 60'].
2026-06-16 10:20:52.137 | INFO     | simulation_1-artifact_manager:82 - Artifact additional filter terms are None.


/mnt/share/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: In a future version of pandas, a length 1 tuple will be returned when iterating over a groupby with a grouper equal to a list of length 1. Don't supply a list with a single grouper to avoid this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/mnt/share/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:234: FutureWarning: In a future version of pandas, a length 1 tuple will be returned when iterating over a groupby with a grouper equal to a list of length 1. Don't supply a list with a single grouper to avoid this warning.
  sub_tables = list(data.groupby(list(other_params)))
/mnt/share/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpo

TypeError: NonLogLinearRiskEffect.validate_rr_data.<locals>.values_are_monotonically_increasing() got an unexpected keyword argument 'include_groups'

## 2) Only severe hemorrhage cases die from hemorrhage

In [ ]:
sim_mort = make_sim('baseline')
run_to_step(sim_mort, SIMULATION_EVENT_NAMES.MORTALITY)

APH_SEVERITY_COL = f"{COLUMNS.ANTEPARTUM_HEMORRHAGE}_severity"
PPH_SEVERITY_COL = f"{COLUMNS.POSTPARTUM_HEMORRHAGE}_severity"

mort_pop = sim_mort.get_population([
    COLUMNS.MOTHER_CAUSE_OF_DEATH,
    APH_SEVERITY_COL,
    PPH_SEVERITY_COL,
])

hemo_cod = [COLUMNS.ANTEPARTUM_HEMORRHAGE, COLUMNS.POSTPARTUM_HEMORRHAGE]
dead_hemo = mort_pop[mort_pop[COLUMNS.MOTHER_CAUSE_OF_DEATH].isin(hemo_cod)]

bad_aph = dead_hemo[
    (dead_hemo[COLUMNS.MOTHER_CAUSE_OF_DEATH] == COLUMNS.ANTEPARTUM_HEMORRHAGE)
    & (dead_hemo[APH_SEVERITY_COL] != HEMORRHAGE_SEVERITY.SEVERE)
]
bad_pph = dead_hemo[
    (dead_hemo[COLUMNS.MOTHER_CAUSE_OF_DEATH] == COLUMNS.POSTPARTUM_HEMORRHAGE)
    & (dead_hemo[PPH_SEVERITY_COL] != HEMORRHAGE_SEVERITY.SEVERE)
]

assert len(bad_aph) == 0, f'{len(bad_aph)} non-severe APH deaths found'
assert len(bad_pph) == 0, f'{len(bad_pph)} non-severe PPH deaths found'
print(f'PASSED: all {len(dead_hemo)} hemorrhage deaths were severe')

2026-06-11 11:40:33.287 | INFO     | simulation_6-artifact_manager:80 - Running simulation from artifact located at /mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/aph/ethiopia.hdf.
2026-06-11 11:40:33.288 | INFO     | simulation_6-artifact_manager:81 - Artifact base filter terms are ['draw == 60'].
2026-06-11 11:40:33.289 | INFO     | simulation_6-artifact_manager:82 - Artifact additional filter terms are None.


/mnt/share/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/mnt/share/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/mnt/share/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/pytho

2026-06-11 11:40:59.994 | WARNING  | simulation_6-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-06-11 11:40:59.995 | WARNING  | simulation_6-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-06-11 11:41:00.106 | WARNING  | simulation_6-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure' during setup.
2026-06-11 11:41:00.108 | WARNING  | simulation_6-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'ensemble_distribution_weights' during setup.
2026-06-11 11:41:00.109 | WARNING  | simulation_6-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure_standard_deviation' during setup.
2026-06-11 11:41:00.111 | WARNING  | simulation_6-lookup_table_manager:85 - 

2026-06-11 11:41:02.098 | INFO     | simulation_6 - vivarium.framework.engine:280 - 2025-01-01 00:00:00
2026-06-11 11:41:13.237 | INFO     | simulation_6 - vivarium.framework.engine:280 - 2025-01-02 00:00:00
2026-06-11 11:41:14.105 | INFO     | simulation_6 - vivarium.framework.engine:280 - 2025-01-03 00:00:00
2026-06-11 11:41:15.476 | INFO     | simulation_6 - vivarium.framework.engine:280 - 2025-01-04 00:00:00
2026-06-11 11:41:27.559 | INFO     | simulation_6 - vivarium.framework.engine:280 - 2025-01-05 00:00:00
2026-06-11 11:41:40.833 | INFO     | simulation_6 - vivarium.framework.engine:280 - 2025-01-06 00:00:00
2026-06-11 11:41:41.397 | WARNING  | simulation_6-population_manager:747 - The 'antepartum_hemorrhage.incidence_risk.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'antepartum_hemorrhage.incidence_risk.paf'.
2026-06-11 11:41:41.482 | INFO     | simulation_6 - vivarium.framewor

## 3) Misoprostol affects postpartum hemorrhage incidence but not antepartum hemorrhage incidence, severe / total incidence are the same bt aph and pph, and the aph/pph incidence fraction matches the artifact postpartum fraction

In [23]:
art_pp_frac = artifact.load("cause.maternal_hemorrhage.postpartum_fraction").mean(axis=1).reset_index()
art_pp_frac_male = art_pp_frac.copy()
art_pp_frac_male["sex"] = "Male"
art_pp_frac = pd.concat([art_pp_frac, art_pp_frac_male]).set_index(["sex", "age_start", "age_end", "year_start", "year_end"]).rename(columns={0: "value"})
pop_structure = artifact.load("population.structure").xs("Ethiopia", level="location").rename(columns={"value": "pop_struc"})
weighted_pp_frac = pd.concat([art_pp_frac, pop_structure], axis=1).dropna()
weighted_pp_frac["pop_struc"] = weighted_pp_frac["pop_struc"] / weighted_pp_frac["pop_struc"].sum()
assert abs(weighted_pp_frac["pop_struc"].sum() - 1) < (.1 ** 6) # rounds to 1
weighted_pp_frac = (weighted_pp_frac["value"] * weighted_pp_frac["pop_struc"]).sum()
weighted_pp_frac


0.5219442318792247

In [20]:
APH_SEVERITY_COL = f"{COLUMNS.ANTEPARTUM_HEMORRHAGE}_severity"
PPH_SEVERITY_COL = f"{COLUMNS.POSTPARTUM_HEMORRHAGE}_severity"

def run_sim_for_pph_summary(scenario: str) -> pd.DataFrame:
    sim = make_sim(scenario)
    run_to_step(sim, SIMULATION_EVENT_NAMES.POSTPARTUM_HEMORRHAGE)



    pop = sim.get_population([
        COLUMNS.POSTPARTUM_HEMORRHAGE,
        COLUMNS.ANTEPARTUM_HEMORRHAGE,
        COLUMNS.MISOPROSTOL_AVAILABLE,
        COLUMNS.DELIVERY_FACILITY_TYPE,
        APH_SEVERITY_COL,
        PPH_SEVERITY_COL,
    ])
    return pop

In [21]:
def pph_summary_for_scenario(scenario: str) -> pd.DataFrame:
    pop = run_sim_for_pph_summary(scenario)

    out = []
    aph = float(pop[COLUMNS.ANTEPARTUM_HEMORRHAGE].mean())
    pph = float(pop[COLUMNS.POSTPARTUM_HEMORRHAGE].mean())
    aph_sev_frac = (pop[APH_SEVERITY_COL].replace({'none': np.nan, 'moderate':'False', 'severe': 'True'}).dropna() == 'True').sum() / pop[COLUMNS.ANTEPARTUM_HEMORRHAGE].sum()
    pph_sev_frac = (pop[PPH_SEVERITY_COL].replace({'none': np.nan, 'moderate':'False', 'severe': 'True'}).dropna() == 'True').sum() / pop[COLUMNS.POSTPARTUM_HEMORRHAGE].sum()
    out.append({
        'scenario': scenario, 
        'group': 'overall', 
        'aph_incidence': aph, 
        'pph_incidence': pph, 
        'aph_incidence_frac': aph / (aph + pph), 
        'pph_incidence_frac': pph / (aph + pph), 
        'aph_sev_frac': aph_sev_frac, 
        'pph_sev_frac': pph_sev_frac,
        'n': len(pop)
        })

    for value in [True, False]:
        sub = pop[pop[COLUMNS.MISOPROSTOL_AVAILABLE] == value]
        if len(sub) == 0:
            continue
        aph = float(sub[COLUMNS.ANTEPARTUM_HEMORRHAGE].mean())
        pph = float(sub[COLUMNS.POSTPARTUM_HEMORRHAGE].mean())
        aph_sev_frac = (sub[APH_SEVERITY_COL].replace({'none': np.nan, 'moderate':'False', 'severe': 'True'}).dropna() == 'True').sum() / sub[COLUMNS.ANTEPARTUM_HEMORRHAGE].sum()
        pph_sev_frac = (sub[PPH_SEVERITY_COL].replace({'none': np.nan, 'moderate':'False', 'severe': 'True'}).dropna() == 'True').sum() / sub[COLUMNS.POSTPARTUM_HEMORRHAGE].sum()
        out.append({
            'scenario': scenario,
            'group': f'misoprostol_available={value}',
            'aph_incidence': aph,
            'pph_incidence': pph,
            'aph_incidence_frac': aph / (aph + pph),  
            'pph_incidence_frac': pph / (aph + pph), 
            'aph_sev_frac': aph_sev_frac,  
            'pph_sev_frac': pph_sev_frac,
            'n': len(sub),
        })

    home = pop[pop[COLUMNS.DELIVERY_FACILITY_TYPE] == 'home']
    if len(home):
        aph = float(home[COLUMNS.ANTEPARTUM_HEMORRHAGE].mean())
        pph = float(home[COLUMNS.POSTPARTUM_HEMORRHAGE].mean())
        aph_sev_frac = (home[APH_SEVERITY_COL].replace({'none': np.nan, 'moderate':'False', 'severe': 'True'}).dropna() == 'True').sum() / home[COLUMNS.ANTEPARTUM_HEMORRHAGE].sum()
        pph_sev_frac = (home[PPH_SEVERITY_COL].replace({'none': np.nan, 'moderate':'False', 'severe': 'True'}).dropna() == 'True').sum() / home[COLUMNS.POSTPARTUM_HEMORRHAGE].sum()
        out.append({
            'scenario': scenario,
            'group': 'home_only',
            'aph_incidence': aph,
            'pph_incidence': pph, 
            'aph_incidence_frac': aph / (aph + pph), 
            'pph_incidence_frac': pph / (aph + pph), 
            'aph_sev_frac': aph_sev_frac, 
            'pph_sev_frac': pph_sev_frac,
            'n': len(home),
        })

    return pd.DataFrame(out)

baseline_pph = pph_summary_for_scenario('baseline')
miso_vv_pph = pph_summary_for_scenario('misoprostol_vv')

check4 = pd.concat([baseline_pph, miso_vv_pph], ignore_index=True)
check4

2026-06-23 06:20:17.001 | INFO     | simulation_5-artifact_manager:80 - Running simulation from artifact located at /mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/aph/ethiopia.hdf.
2026-06-23 06:20:17.002 | INFO     | simulation_5-artifact_manager:81 - Artifact base filter terms are ['draw == 60'].
2026-06-23 06:20:17.003 | INFO     | simulation_5-artifact_manager:82 - Artifact additional filter terms are None.


/mnt/share/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/mnt/share/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/mnt/share/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/pytho

2026-06-23 06:20:43.447 | WARNING  | simulation_5-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-06-23 06:20:43.448 | WARNING  | simulation_5-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-06-23 06:20:43.570 | WARNING  | simulation_5-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure' during setup.
2026-06-23 06:20:43.571 | WARNING  | simulation_5-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'ensemble_distribution_weights' during setup.
2026-06-23 06:20:43.572 | WARNING  | simulation_5-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure_standard_deviation' during setup.
2026-06-23 06:20:43.572 | WARNING  | simulation_5-lookup_table_manager:85 - 

2026-06-23 06:20:46.992 | INFO     | simulation_5 - vivarium.framework.engine:280 - 2025-01-01 00:00:00
2026-06-23 06:21:02.262 | INFO     | simulation_5 - vivarium.framework.engine:280 - 2025-01-02 00:00:00
2026-06-23 06:21:03.777 | INFO     | simulation_5 - vivarium.framework.engine:280 - 2025-01-03 00:00:00
2026-06-23 06:21:06.071 | INFO     | simulation_5 - vivarium.framework.engine:280 - 2025-01-04 00:00:00
2026-06-23 06:21:21.061 | INFO     | simulation_5 - vivarium.framework.engine:280 - 2025-01-05 00:00:00
2026-06-23 06:21:38.947 | INFO     | simulation_5 - vivarium.framework.engine:280 - 2025-01-06 00:00:00
2026-06-23 06:21:40.027 | WARNING  | simulation_5-population_manager:747 - The 'antepartum_hemorrhage.incidence_risk.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'antepartum_hemorrhage.incidence_risk.paf'.
2026-06-23 06:21:40.171 | INFO     | simulation_5 - vivarium.framewor

/mnt/share/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/mnt/share/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/mnt/share/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/pytho

2026-06-23 06:22:17.060 | WARNING  | simulation_6-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-06-23 06:22:17.062 | WARNING  | simulation_6-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-06-23 06:22:17.185 | WARNING  | simulation_6-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure' during setup.
2026-06-23 06:22:17.186 | WARNING  | simulation_6-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'ensemble_distribution_weights' during setup.
2026-06-23 06:22:17.187 | WARNING  | simulation_6-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure_standard_deviation' during setup.
2026-06-23 06:22:17.188 | WARNING  | simulation_6-lookup_table_manager:85 - 

2026-06-23 06:22:20.602 | INFO     | simulation_6 - vivarium.framework.engine:280 - 2025-01-01 00:00:00
2026-06-23 06:22:35.791 | INFO     | simulation_6 - vivarium.framework.engine:280 - 2025-01-02 00:00:00
2026-06-23 06:22:37.330 | INFO     | simulation_6 - vivarium.framework.engine:280 - 2025-01-03 00:00:00
2026-06-23 06:22:39.609 | INFO     | simulation_6 - vivarium.framework.engine:280 - 2025-01-04 00:00:00
2026-06-23 06:22:54.651 | INFO     | simulation_6 - vivarium.framework.engine:280 - 2025-01-05 00:00:00
2026-06-23 06:23:12.726 | INFO     | simulation_6 - vivarium.framework.engine:280 - 2025-01-06 00:00:00
2026-06-23 06:23:13.768 | WARNING  | simulation_6-population_manager:747 - The 'antepartum_hemorrhage.incidence_risk.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'antepartum_hemorrhage.incidence_risk.paf'.
2026-06-23 06:23:13.909 | INFO     | simulation_6 - vivarium.framewor

,scenario,group,aph_incidence,pph_incidence,aph_incidence_frac,pph_incidence_frac,aph_sev_frac,pph_sev_frac,n
0,baseline,overall,0.048733,0.055583,0.467167,0.532833,0.158516,0.162669,120000
1,baseline,misoprostol_available=False,0.048733,0.055583,0.467167,0.532833,0.158516,0.162669,120000
2,baseline,home_only,0.048477,0.101856,0.322465,0.677535,0.154114,0.171258,29581
3,misoprostol_vv,overall,0.048733,0.053158,0.478286,0.521714,0.158516,0.161624,120000
4,misoprostol_vv,misoprostol_available=True,0.048373,0.061019,0.442200,0.557800,0.142132,0.164990,8145
5,misoprostol_vv,misoprostol_available=False,0.048760,0.052586,0.481122,0.518878,0.159699,0.161340,111855
6,misoprostol_vv,home_only,0.048477,0.092019,0.345043,0.654957,0.154114,0.169728,29581


In [25]:
art_sev_frac = artifact.load("cause.maternal_hemorrhage.severe_fraction").mean(axis=1) 
art_sev_frac = art_sev_frac[art_sev_frac != 0].mean()
check4["weighted_pp_frac"] = weighted_pp_frac
check4["art_sev_frac"] = art_sev_frac
targets = pd.concat([check4["weighted_pp_frac"], check4["art_sev_frac"], check4["art_sev_frac"]], axis=1)
targets.columns = ["pph_incidence_frac", "aph_sev_frac", "pph_sev_frac"]
check4 = check4.drop(columns=["weighted_pp_frac", "art_sev_frac"])
targets

,pph_incidence_frac,aph_sev_frac,pph_sev_frac
0,0.521944,0.149366,0.149366
1,0.521944,0.149366,0.149366
2,0.521944,0.149366,0.149366
3,0.521944,0.149366,0.149366
4,0.521944,0.149366,0.149366
5,0.521944,0.149366,0.149366
6,0.521944,0.149366,0.149366


In [30]:
# number of standard errors away from targets
n_std_errors = pd.concat([check4["n"] * (check4["aph_incidence"] + check4["pph_incidence"]), check4["n"] * check4["aph_incidence"], check4["n"] * check4["pph_incidence"]], axis=1)
n_std_errors = n_std_errors.rename(columns={0: "pph_incidence_frac", 1: "aph_sev_frac", 2: "pph_sev_frac"})
check4copy = check4.reset_index().copy().drop(columns=["aph_incidence", "pph_incidence", "aph_incidence_frac", "n", "index"]).set_index(["scenario", "group"])
n_std_errors.index = check4copy.index
std_error = np.sqrt(check4copy * (1 - check4copy) / n_std_errors)
targets.index = check4copy.index
std_error.index = std_error.index
std_error
(check4copy - targets) / std_error

pph_incidence_frac  aph_sev_frac  pph_sev_frac
scenario       group                                                                      
baseline       overall                                2.441763      1.915789      2.943726
               misoprostol_available=False            2.441763      1.915789      2.943726
               home_only                             22.197882      0.498002      3.189668
misoprostol_vv overall                               -0.050911      1.915789      2.659637
               misoprostol_available=True             2.155023     -0.411224      0.938407
               misoprostol_available=False           -0.653412      2.083163      2.496448
               home_only                             18.037942      0.498002      2.829946

In [31]:
check4copy / targets

pph_incidence_frac  aph_sev_frac  pph_sev_frac
scenario       group                                                                      
baseline       overall                                1.020861      1.061256      1.089060
               misoprostol_available=False            1.020861      1.061256      1.089060
               home_only                              1.298099      1.031789      1.146565
misoprostol_vv overall                                0.999559      1.061256      1.082067
               misoprostol_available=True             1.068697      0.951568      1.104601
               misoprostol_available=False            0.994125      1.069180      1.080163
               home_only                              1.254840      1.031789      1.136323

Home is higher. Misoprostol should be lowering pph, this seems backwards. 
Within home, misoprostol_available=True is subset, and pph is lower which is the effect we want. Effect seems going right way.
But home has much higher pph 
confounding path - anc delivery mechanism for ifa. so ppl at home more likely to not have ifa, and have lower hemoglobin. but lower hemoglobin should be increasing risk of aph and pph equally. 
does this table include a/e/m in pop?
IFD ANC correlation like we were looking at last time?
-Could my pp_Frac be wrong for this pop?

drill in on one number that seems weird, how does it correspond to other factors. checking misoprostol true is actually subset of home, what else does it correlate with

2nd order =Misoprostol=true only happens when you survive pregnancy. aph_sev_frac low bc ppl who died from aph are not able to get misoprostol





In [ ]:
# Directional check: compare home-birth PPH incidence across scenarios.
# Misoprostol is only available to home births so that is the right comparison group.
baseline_home = float(check4.query("scenario == 'baseline' and group == 'home_only'")['pph_incidence'].iloc[0])
miso_home = float(check4.query("scenario == 'misoprostol_vv' and group == 'home_only'")['pph_incidence'].iloc[0])

pd.DataFrame([
    {'metric': 'baseline_home_birth_pph_incidence', 'value': baseline_home},
    {'metric': 'misoprostol_vv_home_birth_pph_incidence', 'value': miso_home},
    {'metric': 'directional_pass', 'value': miso_home < baseline_home},
])